# 00e — SymbTr Turkish Makam: Data Preparation

**Thesis context:** SymbTr (Karaosmanoğlu, 2012) is the largest machine-readable collection
of Turkish Makam music, containing 2,200 pieces in 155 makams. MIDI files are available
directly — no transcription is needed. This makes Turkish Makam the only tradition in this
study with native symbolic representation.

**Metadata:** Every filename encodes four fields separated by `--`:
`{makam}--{form}--{usul}--{title}--{composer}.mid`
- **makam**: tonal/modal system (e.g. `rast`, `hicaz`, `saba`, `ussak`) — equivalent to raga
- **usul**: rhythmic cycle (e.g. `duyek`, `aksak`, `sofyan`) — equivalent to tala
- **form**: compositional form (e.g. `sarki`, `pesrev`, `sazsemaisi`)

Additionally, the SymbTr TXT files contain a `Koma53` column encoding pitch in 53-tone equal
temperament commas — the microtonal ground truth that will inform EC-REMI deviation tokens
in Step 5.

**Sampling strategy:** Stratified across makams. The top 20 most frequent makams receive
proportional representation; rarer makams receive at least 1 sample.

**Output:**
- `data/processed/turkish_makam/midi/` — 200 MIDI files
- `data/metadata/symbtr_selected.csv` — per-piece metadata (makam, usul, form, composer)


In [1]:
import sys
from pathlib import Path

# Locate project root regardless of where Jupyter was launched from.
# Searches upward for PROGRESS.md — the root marker.
_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [2]:
import pandas as pd
import shutil
import re

RAW_MIDI = PROJECT_ROOT / "datasets" / "SymbTr" / "midi"
RAW_TXT  = PROJECT_ROOT / "datasets" / "SymbTr" / "txt"
OUT_MIDI = PROJECT_ROOT / "data" / "processed" / "turkish_makam" / "midi"
META_DIR = PROJECT_ROOT / "data" / "metadata"

OUT_MIDI.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

## 1. Parse all filenames → metadata DataFrame

In [3]:
all_midi = sorted(RAW_MIDI.glob("*.mid"))
print(f"Total SymbTr MIDI files: {len(all_midi)}")

records = []
for p in all_midi:
    stem   = p.stem
    parts  = stem.split("--")
    # Filename pattern: makam--form--usul--title--composer
    # Some pieces have fewer components; handle gracefully
    makam    = parts[0] if len(parts) > 0 else "unknown"
    form     = parts[1] if len(parts) > 1 else "unknown"
    usul     = parts[2] if len(parts) > 2 else "unknown"
    title    = parts[3].replace("_", " ") if len(parts) > 3 else stem
    composer = parts[4].replace("_", " ") if len(parts) > 4 else "unknown"

    records.append({
        "filename": p.name,
        "makam"   : makam,
        "form"    : form,
        "usul"    : usul,
        "title"   : title,
        "composer": composer,
        "midi_path": str(p),
    })

df = pd.DataFrame(records)
print(f"Parsed {len(df)} pieces")
print(f"\nUnique makams : {df['makam'].nunique()}")
print(f"Unique usuls  : {df['usul'].nunique()}")
print(f"Unique forms  : {df['form'].nunique()}")
df.head(5)

Total SymbTr MIDI files: 2200
Parsed 2200 pieces

Unique makams : 155
Unique usuls  : 88
Unique forms  : 56


,filename,makam,form,usul,title,composer,midi_path
0,acem--ilahi--duyek--aldanma_dunya--zekai_dede.mid,acem,ilahi,duyek,aldanma dunya,zekai dede,/Users/mohammadashraf/Desktop/Thesis-Best/data...
1,acem--ilahi--nimevsat--calabim_bir--haci_bayra...,acem,ilahi,nimevsat,calabim bir,haci bayram veli,/Users/mohammadashraf/Desktop/Thesis-Best/data...
2,acem--kupe--duyek--zulfunu--ahmet_avni_konuk.mid,acem,kupe,duyek,zulfunu,ahmet avni konuk,/Users/mohammadashraf/Desktop/Thesis-Best/data...
3,acem--selam--devrikebir--asik-i_ger--huseyin_f...,acem,selam,devrikebir,asik-i ger,huseyin fahreddin dede,/Users/mohammadashraf/Desktop/Thesis-Best/data...
4,acem--seyir--sofyan--1--erol_bingol.mid,acem,seyir,sofyan,1,erol bingol,/Users/mohammadashraf/Desktop/Thesis-Best/data...


## 2. Makam frequency distribution

In [4]:
makam_counts = df["makam"].value_counts()
print(f"Makam distribution (top 30 of {len(makam_counts)}):")
print(makam_counts.head(30).to_string())

Makam distribution (top 30 of 155):
makam
hicaz              157
nihavent           130
ussak              118
rast               109
huzzam              96
segah               92
huseyni             92
mahur               88
hicazkar            79
kurdilihicazkar     70
muhayyer            67
saba                66
acemasiran          63
beyati              62
buselik             57
karcigar            53
hicaz_humayun       38
acemkurdi           37
evic                33
muhayyerkurdi       32
tahir               31
nisaburek           26
suzinak_zirgule     26
gerdaniye           26
nikriz              25
sehnaz              25
sultaniyegah        22
yegah               22
neva                21
hisarbuselik        20


In [5]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend — safe for all environments
import matplotlib.pyplot as plt

top_makams = makam_counts.head(25)
fig, ax = plt.subplots(figsize=(12, 5))
top_makams.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Makam")
ax.set_ylabel("Number of pieces")
ax.set_title("SymbTr: Top 25 Makams by Piece Count (full collection, n=2200)")
ax.tick_params(axis="x", rotation=45, labelsize=9)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / "results" / "symbtr_makam_distribution.png"), dpi=150)
plt.show()
print("Chart saved → results/symbtr_makam_distribution.png")

Chart saved → results/symbtr_makam_distribution.png


/var/folders/88/d6f4kns57fz2gzfzhd0dktfw0000gn/T/ipykernel_74082/593718060.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Stratified sample — 200 pieces across makams

Each makam receives at least 1 piece. Makams with more pieces receive proportionally
more, capped to avoid over-representing any single makam.


In [6]:
N = 200

sampled_parts = []
for makam, count in makam_counts.items():
    n_select = max(1, round(N * count / len(df)))
    subset   = df[df["makam"] == makam].sample(
        min(n_select, count), random_state=42
    )
    sampled_parts.append(subset)

sampled = pd.concat(sampled_parts)
if len(sampled) > N:
    sampled = sampled.sample(N, random_state=42)
elif len(sampled) < N:
    remaining = df[~df["filename"].isin(sampled["filename"])]
    topup = remaining.sample(min(N - len(sampled), len(remaining)), random_state=42)
    sampled = pd.concat([sampled, topup])

sampled = sampled.reset_index(drop=True)
print(f"Selected: {len(sampled)} pieces across {sampled['makam'].nunique()} makams")
print(f"\nMakam distribution in selection (top 20):")
print(sampled["makam"].value_counts().head(20).to_string())

Selected: 200 pieces across 117 makams

Makam distribution in selection (top 20):
makam
ussak              10
hicaz               9
rast                8
mahur               8
huseyni             7
nihavent            7
segah               6
beyati              6
muhayyer            5
karcigar            4
buselik             4
hicazkar            4
saba                3
acemasiran          3
muhayyerkurdi       3
tahir               3
hisarbuselik        2
kurdilihicazkar     2
suzidil             2
nisaburek           2


## 4. Copy MIDI files to processed directory

In [7]:
copied_names = []

for i, row in sampled.iterrows():
    src      = Path(row["midi_path"])
    out_name = f"symbtr_{i:03d}_{row['makam']}_{row['form']}.mid"
    dst      = OUT_MIDI / out_name
    shutil.copy2(src, dst)
    copied_names.append(out_name)

print(f"Copied {len(copied_names)} files → data/processed/turkish_makam/midi/")

Copied 200 files → data/processed/turkish_makam/midi/


## 5. Note on 53-TET Koma data in TXT files

The SymbTr TXT files contain a `Koma53` column: pitch expressed in 53-tone equal
temperament (53-TET) commas. This is the theoretical pitch representation used in
Ottoman/Turkish Makam music theory (Arel-Ezgi-Uzdilek system).

This data will be used in Step 5 (EC-REMI design) to calibrate the microtonal deviation
tokens for Turkish Makam — specifically to set the Koma offset values for each makam's
characteristic pitch inflections. A brief preview is shown below.


In [8]:
# Preview the 53-TET data from the TXT file for one selected piece
sample_piece = sampled.iloc[0]
txt_filename = sample_piece["filename"].replace(".mid", ".txt")
txt_path     = RAW_TXT / txt_filename

if txt_path.exists():
    txt_df = pd.read_csv(txt_path, sep="\t", nrows=20, encoding="utf-8")
    print(f"TXT file: {txt_filename}")
    print(f"Columns: {txt_df.columns.tolist()}")
    print("\nFirst 10 note rows:")
    note_rows = txt_df[txt_df["Koma53"].notna() & (txt_df["Koma53"] != "")].head(10)
    print(note_rows[["Sira", "NotaAE", "Koma53", "KomaAE", "Pay", "Payda", "Bas"]].to_string())
else:
    print(f"TXT file not found: {txt_path}")

TXT file: rast--seyir--senginsemai--1--erol_bingol.txt
Columns: ['Sira', 'Kod', 'Nota53', 'NotaAE', 'Koma53', 'KomaAE', 'Pay', 'Payda', 'Ms', 'LNS', 'Bas', 'Soz1', 'Offset']

First 10 note rows:
   Sira NotaAE  Koma53  KomaAE  Pay  Payda  Bas
0     1    NaN       0       0    6      4    0
1     2     G4     296     296    1      4   96
2     3     G4     296     296    1      8   96
3     4     A4     305     305    1      8   96
4     5   B4b1     313     313    3     16   96
5     6     A4     305     305    1     16   96
6     7     D5     327     327    1     16   96
7     8     C5     318     318    1     16   96
8     9   B4b1     313     313    1     16   96
9    10     A4     305     305    1     16   96


## 6. Save metadata and verify

In [9]:
sampled = sampled.copy()
sampled["processed_filename"] = copied_names
sampled.to_csv(META_DIR / "symbtr_selected.csv", index=False)
print("Metadata saved → data/metadata/symbtr_selected.csv")

midi_files = list(OUT_MIDI.glob("*.mid"))
print(f"\nMIDI files in output dir : {len(midi_files)}")
print(f"Makam coverage           : {sampled['makam'].nunique()} makams")
print(f"Form coverage            : {sampled['form'].nunique()} forms")
print(f"Usul coverage            : {sampled['usul'].nunique()} usuls")
print("\n✓ Turkish Makam preparation complete.")

Metadata saved → data/metadata/symbtr_selected.csv

MIDI files in output dir : 200
Makam coverage           : 117 makams
Form coverage            : 25 forms
Usul coverage            : 41 usuls

✓ Turkish Makam preparation complete.
